In [8]:
import sys
sys.path.insert(0, 'src')

from sklearn import set_config
set_config(display='text')
# I really hate this setting. because my desktop is setting in Korean.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier

from final_prj_for_stat486.features import extract_title, add_group_size
from final_prj_for_stat486.models import CompanionFeatures

In [2]:
#Load in the dataset

data = pd.read_csv('data/train.csv')
test_data = pd.read_csv('data/test.csv')

title_args = dict(title_map={'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'},
                  keep=['Mr', 'Mrs', 'Miss', 'Master'], other='Rare')
for df in (data, test_data):
    df['Title'] = extract_title(df['Name'], **title_args)

data, test_data = add_group_size(data, test_data, 'Ticket')

X = data.drop(['Survived', 'Name'], axis=1)
y = data['Survived']

In [3]:
# Preproccesing

companions = CompanionFeatures(group_col='Ticket', id_col='PassengerId', fill=0.5)

numeric_features = ['Pclass', 'Age', 'SibSp', 'Parch', 'Fare',
                    'companion_surv_rate', 'n_known_companions', 'has_known_companion', 'group_size']
categorical_features = ['Sex', 'Embarked', 'Title']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

full_pipeline = ColumnTransformer([
    ('numeric', numeric_pipeline, numeric_features),
    ('categorical', categorical_pipeline, categorical_features)
])

# X_transformed = full_pipeline.fit_transform(X)

def make_model(**gbm_params):
    return Pipeline([
        ('companions', clone(companions)),
        ('prep', clone(full_pipeline)),
        ('model', GradientBoostingClassifier(random_state=42, **gbm_params))
    ])

In [4]:
# Split the dataset into training and testing sets

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# Grid Search for hyperparameter tuning

params = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    # 'model__ccp_alpha': [0.0001, 0.001, 0.01, 0.1, 1]
}

grid = GridSearchCV(estimator=make_model(), 
                    param_grid=params, 
                    cv=5, 
                    n_jobs=-1, 
                    verbose=2)

grid.fit(x_train, y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('companions', CompanionFeatures()),
                                       ('prep',
                                        ColumnTransformer(transformers=[('numeric',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         ['Pclass',
                                                                          'Age',
                                                                          'SibSp',
                                                                          'Parch',
     

In [13]:
gradient_boost = make_model(n_estimators=200, learning_rate=0.05, max_depth=3)
gradient_boost.fit(x_train, y_train)
print("Gradient Boosting Train Accuracy: {:.2f}%".format(gradient_boost.score(x_train, y_train) * 100))
print("Gradient Boosting Test Accuracy: {:.2f}%".format(gradient_boost.score(x_test, y_test) * 100))

Gradient Boosting Train Accuracy: 91.71%
Gradient Boosting Test Accuracy: 83.80%


In [14]:
# Baseline Random Forest Classifier

random_forest = clone(grid.estimator).set_params(
    model=RandomForestClassifier(n_estimators=100, random_state=42))
random_forest.fit(x_train, y_train)
print("Random Forest Classifier Train Accuracy: {:.2f}%".format(random_forest.score(x_train, y_train) * 100))
print("Random Forest Classifier Test Accuracy: {:.2f}%".format(random_forest.score(x_test, y_test) * 100))

Random Forest Classifier Train Accuracy: 98.31%
Random Forest Classifier Test Accuracy: 84.36%


In [15]:
# Grid Search Best Parameters and Accuracy
print("Best Parameters from Grid Search: ", grid.best_params_)
print("Best Cross-Validation Accuracy from Grid Search: {:.2f}%".format(grid.best_score_ * 100))
print("Test Accuracy with Best Parameters from Grid Search: {:.2f}%".format(grid.score(x_test, y_test) * 100))

Best Parameters from Grid Search:  {'model__max_depth': 30, 'model__min_samples_leaf': 2, 'model__min_samples_split': 10, 'model__n_estimators': 200}
Best Cross-Validation Accuracy from Grid Search: 79.64%
Test Accuracy with Best Parameters from Grid Search: 81.56%
